# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention

In [1]:
# Housekeeping

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


In [2]:
# Dates
start_date = "06-27-2025"
end_date = "07-04-2025"
date_range = start_date + "--" + end_date

# Make sure you have the correct paths

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = home + "references/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = originals + "complete/"
metadata_folder = originals + "avian-influenza/metadata/"

## Read Metadata 

In [3]:
# Read metadata

# Get metadata from GitHub repo
os.chdir(metadata_folder)
metadata = pd.read_csv("SraRunTable_automated.csv")
print(len(metadata)) 

# If metadata_normalized.tsv is updated, merge to get collection dates
# os.chdir(saved)
# metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates
# metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata

print(len(metadata)) 
display(metadata)

9899
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
91


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state
9808,SRR34270245,WGS,148.09,123077830,PRJNA1207547,SAMN49684900,Viral,46647242,USDA-NVSL,2025,...,2025-06-27 13:14:37,1,25-017626-001,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25589976,False,NaN,USA
9809,SRR34270246,WGS,148.47,161623240,PRJNA1207547,SAMN49684899,Viral,62260970,USDA-NVSL,2025,...,2025-06-27 13:14:39,1,25-017513-001,SRP557452,NaN,BLOOD SWAB,SRS25589975,False,NaN,USA
9810,SRR34270247,WGS,148.55,84546577,PRJNA1207547,SAMN49684898,Viral,32413712,USDA-NVSL,2025,...,2025-06-27 13:14:38,1,25-017483-023,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25589974,False,NaN,USA
9811,SRR34270248,WGS,147.97,145783713,PRJNA1207547,SAMN49684897,Viral,56129212,USDA-NVSL,2025,...,2025-06-27 13:14:47,1,25-017483-017,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25589973,False,NaN,USA
9812,SRR34270249,WGS,148.66,221383660,PRJNA1207547,SAMN49684896,Viral,85194575,USDA-NVSL,2025,...,2025-06-27 13:14:45,1,25-017483-010,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25589972,False,NaN,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9894,SRR34270141,WGS,148.49,110079404,PRJNA980729,SAMN49684768,Viral,44080477,USDA-NVSL,2025,...,2025-06-27 13:01:47,1,25-015646-001,SRP441379,NaN,OROPHARYNGEAL SWAB,SRS25589880,False,NaN,USA
9895,SRR34270142,WGS,146.95,112507279,PRJNA980729,SAMN49684767,Viral,39407889,USDA-NVSL,2025,...,2025-06-27 13:01:46,1,25-010529-003,SRP441379,NaN,tracheal swab,SRS25589879,False,NaN,USA
9896,SRR34270143,WGS,148.02,142063798,PRJNA980729,SAMN49684766,Viral,49911910,USDA-NVSL,2025,...,2025-06-27 13:01:47,1,25-010529-002,SRP441379,NaN,tracheal swab,SRS25589878,False,NaN,USA
9897,SRR34270144,WGS,147.58,379458191,PRJNA980729,SAMN49684765,Viral,133079770,USDA-NVSL,2025,...,2025-06-27 13:01:52,1,25-010529-001,SRP441379,NaN,cloacal swab,SRS25589877,False,NaN,USA


In [4]:
# Get list of genotypes

os.chdir(references)

genotypes_df = pd.read_excel("genotype_key.xlsx")

genotypes = list(genotypes_df["Genotype"])

print(genotypes)

# genotypes = ["B3.13", "D1.1"]

# genotypes = ["B3.2"] #, "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'B1.1', 'B1.2', 'B1.3', 'B2.1', 'B2.2', 'B3.1', 'B3.2', 'B3.3', 'B3.4', 'B3.5', 'B3.6', 'B4.1', 'B5.1', 'Minor01', 'Minor04', 'Minor07', 'Minor08', 'Minor09', 'Minor10', 'Minor11', 'Minor12', 'Minor13', 'Minor14', 'Minor15', 'Minor16', 'Minor17', 'Minor18', 'Minor19', 'Minor24', 'Minor25', 'Minor26', 'Minor27', 'Minor28', 'Minor29', 'Minor30', 'Minor31', 'Minor32', 'Minor33', 'Minor34', 'Minor35', 'Minor36', 'Minor37', 'Minor38', 'Minor39', 'Minor40', 'Minor41', 'Minor42', 'Minor43', 'Minor44', 'Minor45', 'Minor46', 'Minor47', 'Minor48', 'B3.7', 'Minor50', 'Minor51', 'C1.1', 'Minor52', 'Minor53', 'B3.11', 'Minor55', 'Minor56', 'Minor57', 'Minor58', 'B3.10', 'C2.1', 'Minor60', 'Minor61', 'B3.8', 'Minor62', 'Minor63', 'B3.12', 'Minor65', 'Minor66', 'Minor67', 'B3.9', 'Minor70', 'Minor71', 'B3.13', 'Minor73', 'Minor74', 'Minor75', 'Minor76', 'Minor77', 'Minor78', 'Minor79', 'Minor80', 'Minor81', 'Minor82', 'Minor83', 'Minor84', 'C3.1', 'Minor86', 'Mino

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[geo_location]|[collection_date]|[host_type]|[genotype]

In metadata, we have: host, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name = genbank_mapping.tsv > genbank_name

geo_location = geo_loc_name (abbreviated)-country (abbreviated) e.g. USA-MD

isolate = isolate

collection date (primary) = Collection_Date

collection date = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = H5N1 (hard-coded)

host type = animals_ref.csv (local)

genotype = genoflu_results.tsv > genotype

## Get genotype

In [5]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_results = genoflu_results.rename(columns={"sample" : "Run"}) # Rename so we can merge

metadata = metadata.merge(genoflu_results, on="Run", how="inner") # Add genoflu results to dataframe, excluding runs without results
metadata = metadata[metadata["Genotype"].isin(genotypes)]

print(len(metadata)) 
display(metadata)

80


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,retraction_detection_date_utc,name_state,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
1,SRR34270246,WGS,148.47,161623240,PRJNA1207547,SAMN49684899,Viral,62260970,USDA-NVSL,2025,...,NaN,USA,2025-06-28_06-29-39,SRR34270246.fa,D1.1,"PB1:ea3, PA:am4, HA:ea3, MP:ea3, NA:am4N1, PB2...","ea3:22-013001-001:PB1, am4:24-030039-001:PA, e...","99.34%, 99.66%, 99.47%, 100.00%, 99.48%, 99.65...","12, 6, 9, 0, 6, 8, 9, 4",Ran on FASTA - No Coverage Report
4,SRR34270249,WGS,148.66,221383660,PRJNA1207547,SAMN49684896,Viral,85194575,USDA-NVSL,2025,...,NaN,USA,2025-06-28_06-29-38,SRR34270249.fa,D1.1,"PB1:ea3, NS:ea3, PB2:am24, NA:am4N1, HA:ea3, P...","ea3:22-013001-001:PB1, ea3:22-013001-001:NS, a...","99.16%, 98.93%, 99.56%, 99.04%, 99.47%, 99.44%...","19, 9, 10, 10, 9, 12, 7, 0",Ran on FASTA - No Coverage Report
7,SRR34270252,WGS,146.56,305934420,PRJNA1207547,SAMN49684893,Viral,108219767,USDA-NVSL,2024,...,NaN,USA,2025-06-28_06-29-38,SRR34270252.fa,C3.1,"PB1:am10, MP:ea1, NP:am5, HA:ea2, PA:ea1, PB2:...","am10:22-038433-002:PB1, ea1:22-003707-003:MP, ...","98.42%, 99.19%, 98.80%, 99.71%, 99.67%, 99.78%...","36, 8, 18, 5, 7, 5, 3, 2",Ran on FASTA - No Coverage Report
8,SRR34270253,WGS,146.59,191484632,PRJNA1207547,SAMN49684910,Viral,64710745,USDA-NVSL,2024,...,NaN,USA,2025-06-28_06-29-38,SRR34270253.fa,B3.2,"PA:ea1, NS:am1.1, HA:ea1, PB1:am1.2, NA:ea1, P...","ea1:22-003707-003:PA, am1.1:22-010085-001:NS, ...","99.72%, 99.76%, 99.71%, 100.00%, 99.79%, 99.74...","6, 2, 5, 0, 3, 6, 2, 1",Ran on FASTA - No Coverage Report
9,SRR34270254,WGS,147.22,172959588,PRJNA1207547,SAMN49684909,Viral,59423016,USDA-NVSL,2024,...,NaN,USA,2025-06-28_06-29-38,SRR34270254.fa,C2.1,"PB2:am19, PA:am3, PB1:am20, NA:ea2, MP:ea1, NP...","am19:23-036193-005:PB2, am3:23-036657-001:PA, ...","99.65%, 99.44%, 99.60%, 99.79%, 99.39%, 99.51%...","8, 12, 9, 3, 6, 6, 12, 7",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,SRR34270140,WGS,148.74,131194009,PRJNA980729,SAMN49684769,Viral,52071639,USDA-NVSL,2025,...,NaN,USA,2025-06-28_06-29-39,SRR34270140.fa,D1.1,"PA:am4, PB1:ea3, NP:am13, NA:am4N1, HA:ea3, NS...","am4:24-030039-001:PA, ea3:22-013001-001:PB1, a...","99.60%, 99.34%, 99.80%, 99.13%, 99.24%, 99.17%...","7, 15, 3, 10, 13, 7, 7, 1",Ran on FASTA - No Coverage Report
86,SRR34270141,WGS,148.49,110079404,PRJNA980729,SAMN49684768,Viral,44080477,USDA-NVSL,2025,...,NaN,USA,2025-06-28_06-29-38,SRR34270141.fa,D1.1,"NP:am13, HA:ea3, NA:am4N1, PB1:ea3, PA:am4, MP...","am13:24-030039-001:NP, ea3:22-013001-001:HA, a...","99.80%, 99.24%, 99.29%, 99.34%, 99.54%, 99.90%...","3, 13, 8, 15, 8, 1, 5, 7",Ran on FASTA - No Coverage Report
87,SRR34270142,WGS,146.95,112507279,PRJNA980729,SAMN49684767,Viral,39407889,USDA-NVSL,2025,...,NaN,USA,2025-06-28_06-29-39,SRR34270142.fa,D1.1,"NS:ea3, MP:ea3, PA:am4, NP:am13, PB1:ea3, HA:e...","ea3:22-013001-001:NS, ea3:22-013001-001:MP, am...","98.81%, 100.00%, 99.89%, 99.73%, 99.33%, 99.35...","10, 0, 2, 4, 12, 11, 5, 8",Ran on FASTA - No Coverage Report
88,SRR34270143,WGS,148.02,142063798,PRJNA980729,SAMN49684766,Viral,49911910,USDA-NVSL,2025,...,NaN,USA,2025-06-28_06-29-39,SRR34270143.fa,D1.1,"NA:am4N1, NP:am13, HA:ea3, PB2:am24, MP:ea3, P...","am4N1:24-030039-001:NA, am13:24-030039-001:NP,...","99.38%, 99.73%, 99.41%, 99.65%, 100.00%, 99.81...","7, 4, 10, 8, 0, 4, 11, 13",Ran on FASTA - No Coverage Report


## Get specific geolocation

In [6]:
# Get specific geolocation and name_state from genbank_mapping.tsv
os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping = genbank_mapping.rename(columns={"sra_run": "Run"}) # Rename so we can merge
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates -- there are ~8 copies of each run
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# Merge with metadata so that we can have specific geolocation
metadata_genbank = pd.concat([metadata, genbank_mapping], join="inner") # Exclude runs without geolocation

# print(metadata_genbank)

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")
# Format: USA-[state abbreviation], e.g. USA-MD
metadata_genbank["Geo_Location"] = metadata_genbank["name_state"].apply(lambda x: 
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(", ", " ").split(" "))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(", ", " ").split(" "))), regex=True).any() 
                                                        # If "x" has neither the state abbreviation nor the full state name
                                                        else 
                                                        x)

# Rename variable back to metadata as we merge metadata and metadata_genbank
metadata = metadata.merge(metadata_genbank, on="Run")

display(metadata) 

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,name_state_y,Geo_Location
0,SRR34270246,WGS,148.47,161623240,PRJNA1207547,SAMN49684899,Viral,62260970,USDA-NVSL,2025,...,2025-06-28_06-29-39,SRR34270246.fa,D1.1,"PB1:ea3, PA:am4, HA:ea3, MP:ea3, NA:am4N1, PB2...","ea3:22-013001-001:PB1, am4:24-030039-001:PA, e...","99.34%, 99.66%, 99.47%, 100.00%, 99.48%, 99.65...","12, 6, 9, 0, 6, 8, 9, 4",Ran on FASTA - No Coverage Report,USA,USA
1,SRR34270249,WGS,148.66,221383660,PRJNA1207547,SAMN49684896,Viral,85194575,USDA-NVSL,2025,...,2025-06-28_06-29-38,SRR34270249.fa,D1.1,"PB1:ea3, NS:ea3, PB2:am24, NA:am4N1, HA:ea3, P...","ea3:22-013001-001:PB1, ea3:22-013001-001:NS, a...","99.16%, 98.93%, 99.56%, 99.04%, 99.47%, 99.44%...","19, 9, 10, 10, 9, 12, 7, 0",Ran on FASTA - No Coverage Report,USA,USA
2,SRR34270252,WGS,146.56,305934420,PRJNA1207547,SAMN49684893,Viral,108219767,USDA-NVSL,2024,...,2025-06-28_06-29-38,SRR34270252.fa,C3.1,"PB1:am10, MP:ea1, NP:am5, HA:ea2, PA:ea1, PB2:...","am10:22-038433-002:PB1, ea1:22-003707-003:MP, ...","98.42%, 99.19%, 98.80%, 99.71%, 99.67%, 99.78%...","36, 8, 18, 5, 7, 5, 3, 2",Ran on FASTA - No Coverage Report,USA,USA
3,SRR34270253,WGS,146.59,191484632,PRJNA1207547,SAMN49684910,Viral,64710745,USDA-NVSL,2024,...,2025-06-28_06-29-38,SRR34270253.fa,B3.2,"PA:ea1, NS:am1.1, HA:ea1, PB1:am1.2, NA:ea1, P...","ea1:22-003707-003:PA, am1.1:22-010085-001:NS, ...","99.72%, 99.76%, 99.71%, 100.00%, 99.79%, 99.74...","6, 2, 5, 0, 3, 6, 2, 1",Ran on FASTA - No Coverage Report,USA,USA
4,SRR34270254,WGS,147.22,172959588,PRJNA1207547,SAMN49684909,Viral,59423016,USDA-NVSL,2024,...,2025-06-28_06-29-38,SRR34270254.fa,C2.1,"PB2:am19, PA:am3, PB1:am20, NA:ea2, MP:ea1, NP...","am19:23-036193-005:PB2, am3:23-036657-001:PA, ...","99.65%, 99.44%, 99.60%, 99.79%, 99.39%, 99.51%...","8, 12, 9, 3, 6, 6, 12, 7",Ran on FASTA - No Coverage Report,USA,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,SRR34270140,WGS,148.74,131194009,PRJNA980729,SAMN49684769,Viral,52071639,USDA-NVSL,2025,...,2025-06-28_06-29-39,SRR34270140.fa,D1.1,"PA:am4, PB1:ea3, NP:am13, NA:am4N1, HA:ea3, NS...","am4:24-030039-001:PA, ea3:22-013001-001:PB1, a...","99.60%, 99.34%, 99.80%, 99.13%, 99.24%, 99.17%...","7, 15, 3, 10, 13, 7, 7, 1",Ran on FASTA - No Coverage Report,USA,USA
76,SRR34270141,WGS,148.49,110079404,PRJNA980729,SAMN49684768,Viral,44080477,USDA-NVSL,2025,...,2025-06-28_06-29-38,SRR34270141.fa,D1.1,"NP:am13, HA:ea3, NA:am4N1, PB1:ea3, PA:am4, MP...","am13:24-030039-001:NP, ea3:22-013001-001:HA, a...","99.80%, 99.24%, 99.29%, 99.34%, 99.54%, 99.90%...","3, 13, 8, 15, 8, 1, 5, 7",Ran on FASTA - No Coverage Report,USA,USA
77,SRR34270142,WGS,146.95,112507279,PRJNA980729,SAMN49684767,Viral,39407889,USDA-NVSL,2025,...,2025-06-28_06-29-39,SRR34270142.fa,D1.1,"NS:ea3, MP:ea3, PA:am4, NP:am13, PB1:ea3, HA:e...","ea3:22-013001-001:NS, ea3:22-013001-001:MP, am...","98.81%, 100.00%, 99.89%, 99.73%, 99.33%, 99.35...","10, 0, 2, 4, 12, 11, 5, 8",Ran on FASTA - No Coverage Report,USA,USA
78,SRR34270143,WGS,148.02,142063798,PRJNA980729,SAMN49684766,Viral,49911910,USDA-NVSL,2025,...,2025-06-28_06-29-39,SRR34270143.fa,D1.1,"NA:am4N1, NP:am13, HA:ea3, PB2:am24, MP:ea3, P...","am4N1:24-030039-001:NA, am13:24-030039-001:NP,...","99.38%, 99.73%, 99.41%, 99.65%, 100.00%, 99.81...","7, 4, 10, 8, 0, 4, 11, 13",Ran on FASTA - No Coverage Report,USA,USA


## Collection Dates

If date is N/A, try finding it first. If a csv file of saved dates (NOT metadata_normalized.tsv) are available, do NOT run the next cell. Comment it out and run the cell after. 

In [7]:

# # Get all dates
# metadata["Collection_Date_Specific"] = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata) if "-" not in x else x) # Real dates have dashes
# # Convert dates to date format
# try:
#     metadata["Collection_Date_Specific"] = metadata["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x))
# except:
#     print("Unable to parse collection date.")

If saved dates are available, un-comment and run the next cell

In [8]:
# Upload saved data -- if doing this, make sure the above cell is commented out
os.chdir(temp_files)
metadata_genbank = pd.read_csv("metadata_genbank_" + date_range + ".csv")

os.chdir(temp_files)

# Get only updated dates

def find_unknown_dates(x, df):
    try:
        date = metadata[metadata["BioSample"] == x]["Collection_Date"].values[0]
        date = dateutil.parser.parse(date, default=datetime(2000, 1, 1)) # Default is January 1st, 2000
        if date.day == dateutil.parser.parse("1/1/2000").day and date.month == dateutil.parser.parse("1/1/2000").month: # If the date autocompleted to default 1/1
            print("year only")
            date = search_collection_date(x, df)
        print("Success", date)
    except:
        date = search_collection_date(x, df) # If it's not parseable as a date

    # Ensure that the date is converted to date format
    try:
        date = dateutil.parser.parse(date, default=datetime(2000, 1, 1))
        if date.day == dateutil.parser.parse("1/1/2000").day and date.month == dateutil.parser.parse("1/1/2000").month: # If the date autocompleted to default 1/1
            date = date.year.strftime("%Y") # Preserve only year
        else: # If actual date
            date = dateutil.parser.parse(date).strftime("%Y-%m-%d")
    except:
        print("Unable to parse date.")

    return date # "If" statement in lambda function will search for the "just year" values

updated_dates = metadata["BioSample"].apply(lambda x: find_unknown_dates(x, metadata)) # Update unknown dates, if possible
metadata["Collection_Date_Specific"] = updated_dates

year only
Unable to find collection date.
Success 2025
Unable to parse date.
year only
Unable to find collection date.
Success 2025
Unable to parse date.
year only
Unable to find collection date.
Success 2024
Unable to parse date.
year only
Unable to find collection date.
Success 2024
Unable to parse date.
year only
Unable to find collection date.
Success 2024
Unable to parse date.
year only
Unable to find collection date.
Success 2024
Unable to parse date.
year only
Unable to find collection date.
Success 2024
Unable to parse date.
year only
Unable to find collection date.
Success 2024
Unable to parse date.
year only
Unable to find collection date.
Success 2025
Unable to parse date.
year only
Unable to find collection date.
Success 2025
Unable to parse date.
year only
Unable to find collection date.
Success 2025
Unable to parse date.
year only
Unable to find collection date.
Success 2025
Unable to parse date.
year only
Unable to find collection date.
Success 2024
Unable to parse date.

In [9]:
# Save the above so we don't have to do it again
os.chdir(temp_files)
metadata.to_csv("metadata_genbank_" + date_range + ".csv")

# Get years from collection dates
metadata["years"] = metadata["Collection_Date_Specific"].apply(lambda x: x.year) # Get year only from collection date

## Get host type

In [10]:
# create a mask, where is True if the host does not exist
mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, 
                            metadata["isolate"].apply(lambda x: 
                                                      x if x != x # If NaN
                                                      or "/" not in x # If split isolate doesn't exist 
                                                      or len(x.split("/")) < 2 # If split isolate is too short
                                                      else x.split("/")[1]), metadata["Host"]) # Provided that we have a long enough isolate with "/" in them, get the host

metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x) # make sure all characters are lowercase

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(home + "references/")
animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2
common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str: # If animal exists in dataframe and isn't NaN
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe, but just in case
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    for i in range(number_of_times_to_add_nan):
        empty_rows = pd.DataFrame(np.nan, index=range(number_of_times_to_add_nan), columns=animals_df.columns)
        animals_df = pd.concat([animals_df, empty_rows], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

[]
                    avian               cattle        feline   other_mammal  \
0        great_horned_owl            dairy_cow           cat     deer mouse   
1            common_raven               cattle  domestic_cat    house_mouse   
2           cooper's_hawk  cattle milk product     feral_cat          skunk   
3            coopers_hawk          bovine_milk        feline  striped_skunk   
4                 peafowl              bovine   domestic-cat     norway rat   
..                    ...                  ...           ...            ...   
799         harris's_hawk                  NaN           NaN            NaN   
800         eurasian_coot                  NaN           NaN            NaN   
801                 layer                  NaN           NaN            NaN   
802            perdicinae                  NaN           NaN            NaN   
803  great-tailed grackle                  NaN           NaN            NaN   

          human         other  new  
0    washin

In [11]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

## Make names using all the attributes we collected

In [12]:
# Make names

metadata = metadata.fillna("") # Make sure the entire name does not become "NaN"

names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata["name_state_y"] + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date_Specific"].apply(lambda x: str(x.strftime("%Y")) if x.month == datetime(2000, 1, 1).month and x.day == datetime(2000, 1, 1).day else x.strftime("%Y")) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

display(metadata["Name"])

0     >SRR34270246|A/red-tailed_hawk/USA/25-017513-0...
1     >SRR34270249|A/great-tailed_grackle/USA/25-017...
2     >SRR34270252|A/bottlenose_dolphin/USA/24-02894...
3     >SRR34270253|A/red_fox/USA/24-020141-001/2024|...
4     >SRR34270254|A/red_fox/USA/24-017742-001/2024|...
                            ...                        
75    >SRR34270140|A/duck/USA/25-015646-002/2025|H5N...
76    >SRR34270141|A/chicken/USA/25-015646-001/2025|...
77    >SRR34270142|A/chicken/USA/25-010529-003/2025|...
78    >SRR34270143|A/chicken/USA/25-010529-002/2025|...
79    >SRR34270144|A/duck/USA/25-010529-001/2025|H5N...
Name: Name, Length: 80, dtype: object

In [13]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="Run", keep="first")

In [14]:
print(metadata)
# metadata.to_csv("metadata_test.csv")

            Run Assay Type  AvgSpotLen      Bases    BioProject     BioSample  \
0   SRR34270246        WGS      148.47  161623240  PRJNA1207547  SAMN49684899   
1   SRR34270249        WGS      148.66  221383660  PRJNA1207547  SAMN49684896   
2   SRR34270252        WGS      146.56  305934420  PRJNA1207547  SAMN49684893   
3   SRR34270253        WGS      146.59  191484632  PRJNA1207547  SAMN49684910   
4   SRR34270254        WGS      147.22  172959588  PRJNA1207547  SAMN49684909   
..          ...        ...         ...        ...           ...           ...   
75  SRR34270140        WGS      148.74  131194009   PRJNA980729  SAMN49684769   
76  SRR34270141        WGS      148.49  110079404   PRJNA980729  SAMN49684768   
77  SRR34270142        WGS      146.95  112507279   PRJNA980729  SAMN49684767   
78  SRR34270143        WGS      148.02  142063798   PRJNA980729  SAMN49684766   
79  SRR34270144        WGS      147.58  379458191   PRJNA980729  SAMN49684765   

   BioSampleModel      Byte

## Make FASTA files

In [15]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

# Create pairs of genotypes and segments, e.g. B3.13_HA
for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [16]:
# Create fasta files 

os.chdir(originals + "complete/")
names = []
for pair in fasta_files.keys():
    output_path = originals + "complete/" + pair + "_" + date_range + "_andersen.fasta"

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names)/8)

>SRR34270263|A/red_fox/USA/24-028269-001/2024|H5N1|USA|2024|other_mammal|A3
>SRR34270263|A/red_fox/USA/24-028269-001/2024|H5N1|USA|2024|other_mammal|A3
>SRR34270263|A/red_fox/USA/24-028269-001/2024|H5N1|USA|2024|other_mammal|A3
>SRR34270263|A/red_fox/USA/24-028269-001/2024|H5N1|USA|2024|other_mammal|A3
>SRR34270263|A/red_fox/USA/24-028269-001/2024|H5N1|USA|2024|other_mammal|A3
>SRR34270263|A/red_fox/USA/24-028269-001/2024|H5N1|USA|2024|other_mammal|A3
>SRR34270263|A/red_fox/USA/24-028269-001/2024|H5N1|USA|2024|other_mammal|A3
>SRR34270263|A/red_fox/USA/24-028269-001/2024|H5N1|USA|2024|other_mammal|A3
>SRR34270253|A/red_fox/USA/24-020141-001/2024|H5N1|USA|2024|other_mammal|B3.2
>SRR34270253|A/red_fox/USA/24-020141-001/2024|H5N1|USA|2024|other_mammal|B3.2
>SRR34270253|A/red_fox/USA/24-020141-001/2024|H5N1|USA|2024|other_mammal|B3.2
>SRR34270253|A/red_fox/USA/24-020141-001/2024|H5N1|USA|2024|other_mammal|B3.2
>SRR34270253|A/red_fox/USA/24-020141-001/2024|H5N1|USA|2024|other_mammal|B3.2
>S